In [ ]:
import json
import os
import logging

def extract_text_from_json_files(directory, section_types):
    logging.basicConfig(level=logging.INFO)
    logging.info(f"Extracting text from JSON files in {directory}")

    for filename in os.listdir(directory):
        if filename.endswith(".json"):
            logging.info(f"Processing file {filename}")
            try:
                with open(os.path.join(directory, filename), 'r') as file:
                    data = json.load(file)
                    base_filename = os.path.splitext(filename)[0]
                    
                    # Check if 'SUPPL' section is present
                    suppl_present = False
                    for passage in data[0]['documents'][0]['passages']:
                        if passage['infons']['section_type'] == 'SUPPL':
                            suppl_present = True
                            break
                    
                    if not suppl_present:
                        for section_type in section_types:
                            extracted_text = ""
                            for passage in data[0]['documents'][0]['passages']:
                                if passage['infons']['section_type'] == section_type:
                                    extracted_text += passage['text'] + " "
                            output_filename = f"{base_filename}_{section_type}.txt"
                            with open(os.path.join(directory, output_filename), 'w', encoding='utf-8', errors='replace') as f:
                                f.write(extracted_text.strip())
                    else:
                        logging.info(f"Skipping file {filename} as it has a 'SUPPL' section")
            except json.JSONDecodeError as e:
                logging.error(f"Error parsing JSON file {filename}: {e}")
            except KeyError as e:
                logging.error(f"Error extracting text from JSON file {filename}: {e}")

if __name__ == "__main__":
    directory = "/Articles with no suppL"
    section_types = ['TITLE','ABSTRACT','INTRO', 'METHODS', 'RESULTS', 'CONCL', 'REF', 'FIG', 'DISCUSS']
    extract_text_from_json_files(directory, section_types)